# Pandas = Labeled, column-oriented data on top of NumPy


### 1. Key differences from NumPy:

- Data is **labeled** (rows + columns have names)
- Columns can have **different dtypes**
- Operations align data by labels, not position

### Mental shift from “array of rows” to “collection of named columns”

### 2. Core Objects: Series & DataFrame
#### Series (1D)

**Think of it as:**

- One column of data
- A NumPy array with labels
- Values → NumPy array
- Index → labels

In [2]:
import pandas as pd

s = pd.Series([100, 200, 300], index=["A", "B", "C"])
print(s.values)
print(s.index)

[100 200 300]
Index(['A', 'B', 'C'], dtype='str')


####  DataFrame (2D)
**Think of a DataFrame as:**

- Dict of Series sharing the same index
- SQL table / CSV in memory

In [ ]:
df = pd.DataFrame({
    "revenue": [1000, 2000, 1500],
    "cost": [400, 800, 600],
    "country": ["US", "IN", "US"]
})

'''
Internally:
{
  "revenue": Series,
  "cost": Series,
  "country": Series
}

'''

### 3. The Index (Most Misunderstood Part)
***What is an Index?***

- Labels for rows
- NOT just row numbers
- Can be anything: ints, strings, dates

**Why Index exists**

- Fast lookups
- Data alignment
- Joins & time series

In [ ]:
# Example(Alignment)
a = pd.Series([1, 2, 3], index=["x", "y", "z"])
b = pd.Series([10, 20, 30], index=["y", "z", "w"])

a + b

# Prevents silent bugs
# Makes joins safer than raw NumPy

w     NaN
x     NaN
y    12.0
z    23.0
dtype: float64

### 4. Columns Are Series (Key Insight)
**Below example explains why**

- Why .mean() works

- Why vectorized math works

- Why .apply() exists


In [9]:
df["revenue"]   # returns a Series
type(df["revenue"])

df["profit"] = df["revenue"] - df["cost"]

### 5. Row vs Column Thinking (Critical for Performance)

**Why this matters**

- Column ops are NumPy-fast

-  Row ops are Python-slow

In [ ]:
# Bad row wise operations/loops
for _, row in df.iterrows():
    row["revenue"] - row["cost"]

# Good Column-wise operations
df["profit"] = df["revenue"] - df["cost"]

### 6. pandas in Real Pipelines (Big Picture)
**In Data Science / ML**
- raw data → pandas → clean features → sklearn

**In Backend Analytics**
- logs → pandas → aggregations → metrics / DB

**pandas is:**

- Not a database
- Not a visualization tool
- A data preparation engine

In [ ]:
df = pd.DataFrame({
    "A": [1, 2],
    "B": [10, 20]
}, index=["x", "y"])

df["C"] = df["A"] + df["B"]

# What is the type of df["C"]? Series

# Why does this operation not need a loop? it performs vectorised operations on columns

# What happens if index labels don’t match? addition will not be performed ans will ne NaN for unmatched index

print(type(df["C"]))

<class 'pandas.Series'>


**Common Mistakes to Avoid**

- Treating pandas like NumPy with labels
- Ignoring the index
- Writing loops too early
- Overusing .apply()

## iloc, loc for series and dataframe

### 1. Index, Labels, Positions (Core Definitions)
- Index: An Index is a **collection of labels** attached to **an axis**.

- Label: A label is a **name** used to **identify data (not its position)**.

- Position: A position is the physical location: 0, 1, 2, ...

- **Key rule**

    ***Labels identify → positions locate***

### 2. Default Index

- When you don’t specify an index, pandas creates: --> Index([0, 1, 2, ...])

⚠️ These are labels, not positions — they just look like positions. 

## 3. Series (1D)
- Structure: One axis, One Index (labels), One value per label

- s = pd.Series([10, 20, 30])

| Label (Index) | Position | Value |
| ------------- | -------- | ----- |
| 0             | 0        | 10    |
| 1             | 1        | 20    |
| 2             | 2        | 30    |


| Expression     | Meaning                        |
| -------------- | ------------------------------ |
| `s.loc[label]` | Value at **label**             |
| `s.iloc[pos]`  | Value at **position**          |
| `s[x]`         | **Label lookup (if label exists)** else error will be thrown, mentioned in class notebook |
| `s[a:b]`       | Positional slice               |


In [ ]:
# by default index is 0, 1, 2
### Example:

s = pd.Series([10, 20, 30])
print(s.loc[1])    # label 1 → 20 
print(s.iloc[1])   # position 1 → 20
print(s[0:2])      # first two elements
print(s[[0,2]])    #  0th and 2nd element

20
20
0    10
1    20
dtype: int64
0    10
2    30
dtype: int64


### 4. DataFrame (2D)
- Structure: Two axes, Row Index (labels), Column Index (labels)

In [ ]:
df = pd.DataFrame(
    {"A": [10, 20, 30],
    "B": [40, 50, 60]}
)

| Row label | Position | A  | B  |
| --------- | -------- | -- | -- |
| 0         | 0        | 10 | 40 |
| 1         | 1        | 20 | 50 |
| 2         | 2        | 30 | 60 |


### DataFrame Indexing Summary
-  .iloc (position-based)
-  df.iloc[row_pos, col_pos]

| Example         | Meaning         |
| --------------- | --------------- |
| `df.iloc[0]`    | First row       |
| `df.iloc[:, 0]` | First column    |
| `df.iloc[1, 1]` | Row 1, column 1 |


### df.loc[row_label, col_label]

#### df.loc[row_label, col_label]

#### **Default row index** is always **auto-generated integers**; **default column index** comes **from your data**, or is auto-generated integers if missing.


| Example          | Meaning            |
| ---------------- | ------------------ |
| `df.loc[0]`      | Row with label `0` |
| `df.loc[:, "A"]` | Column `"A"`       |
| `df.loc[1, "B"]` | Label-based cell   |

✔ Slice end is inclusive

#### df[...] special rule

| Expression         | Meaning                             |
| ------------------ | ----------------------------------- |
| `df["A"]`          | Column `"A"`                        |
| `df[[ "A", "B" ]]` | Multiple columns                    |
| `df[0]`            | ❌ Error (unless column name is `0`) |

##### df[...] never selects rows

#### 5. .loc vs .iloc — One Table to Rule Them All

| Feature      | `.loc`        | `.iloc`       |
| ------------ | ------------- | ------------- |
| Uses         | Labels        | Positions     |
| Index type   | Any           | Integers only |
| Slice end    | Inclusive     | Exclusive     |
| Boolean mask | Label-aligned | Positional    |
| Series       | Yes           | Yes           |
| DataFrame    | Yes           | Yes           |



### 6. Key Things:
**Series**: Index labels identify values

**DataFrame**: Row labels identify rows, Column labels identify columns

**Indexing**: 
- .loc → names
- .iloc → numbers
- [] → context-dependent (avoid for rows)

### 7. One-Line Golden Rule

- If you know the name → use .loc
- If you know the number → use .iloc

I

#### For a dataframe/Series index= ["a", "b", "c"] 
- considering there are 3 rows, here **'index'** will always assign labels to row so we can use df.loc["a"]
- Assignmed index will be used by loc, **iloc** still uses default integer index by pandas 0,1,2..